In [12]:
import pandas as pd
import geopandas as gpd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler, FunctionTransformer, PolynomialFeatures, RobustScaler
from sklearn.linear_model import LinearRegression, Ridge
import os
import fiona
from geopy.distance import geodesic

In [13]:
meteo = pd.read_csv("../departement-78-yvelines/data/meteostat/meteostat.csv")
yvelines_data = gpd.read_file("../datasets_par_departement/departement-78-yvelines-original.geojson")
firepoint_3 = gpd.read_file("../data/hexagones_firepoint_3.geojson")

In [4]:
def calculer_duree(data, colonne_debut, colonne_fin):
    data[colonne_debut] = pd.to_datetime(data[colonne_debut], errors='coerce')
    data[colonne_fin] = pd.to_datetime(data[colonne_fin], errors='coerce')

    data['duree'] = data[colonne_fin] - data[colonne_debut]

    return data

def cyclical_encoding(X, max_value):
    return np.column_stack((
        np.sin(2 * np.pi * X / max_value),
        np.cos(2 * np.pi * X / max_value)
    ))

def season_encoding(month):
    if month in [12, 1, 2]:
        return 0  # Hiver
    elif month in [3, 4, 5]:
        return 1  # Printemps
    elif month in [6, 7, 8]:
        return 2  # Été
    else:
        return 3  # Automne

In [5]:
yvelines_data = calculer_duree(yvelines_data, 'date_debut', 'date_fin')
yvelines_data['centroid'] = yvelines_data.geometry.centroid
yvelines_data['centroid_lon'] = yvelines_data['centroid'].x
yvelines_data['centroid_lat'] = yvelines_data['centroid'].y
yvelines_data['date_debut'] = pd.to_datetime(yvelines_data['date_debut'], errors='coerce')
yvelines_data['hour'] = yvelines_data['date_debut'].dt.hour
yvelines_data['date'] = pd.to_datetime(yvelines_data['date'], errors='coerce')
yvelines_data['month'] = yvelines_data['date'].dt.month
yvelines_data['weekday'] = yvelines_data['date'].dt.weekday
yvelines_data['season'] = yvelines_data['month'].apply(season_encoding)
yvelines_data[['hour_sin', 'hour_cos']] = cyclical_encoding(yvelines_data['hour'], 24)
yvelines_data[['month_sin', 'month_cos']] = cyclical_encoding(yvelines_data['month'], 12)
yvelines_data[['weekday_sin', 'weekday_cos']] = cyclical_encoding(yvelines_data['weekday'], 7)

<ipython-input-5-7905ed1a205d>:2: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  yvelines_data['centroid'] = yvelines_data.geometry.centroid


In [6]:
meteo['id_meteo'] = range(1, len(meteo) + 1)

meteo.head

<bound method NDFrame.head of         index     creneau  temp  dwpt   rhum  prcp  snow   wdir      wspd  \
0          36  2016-01-01   6.0   6.0  100.0   0.0   0.0  130.0  3.111111   
1          36  2016-01-01   7.3   6.7   96.0   0.0   0.0  140.0  5.694444   
2          36  2016-01-01   7.0   5.9   93.0   0.0   0.0  150.0  4.611111   
3          36  2016-01-01   6.0   6.0  100.0   0.0   0.0  130.0  3.111111   
4          36  2016-01-01   5.5   5.5  100.0   0.0   0.0  140.0  3.111111   
...       ...         ...   ...   ...    ...   ...   ...    ...       ...   
207829   2892  2024-06-28  19.0   8.9   52.0   0.0   0.0  280.0  2.500000   
207830   2892  2024-06-28  20.2  10.0   52.0   0.0   0.0  330.0  3.111111   
207831   2892  2024-06-28  20.0   9.0   49.0   0.0   0.0  310.0  3.611111   
207832   2892  2024-06-28  20.0   9.0   49.0   0.0   0.0  310.0  3.611111   
207833   2892  2024-06-28  20.0   9.0   49.0   0.0   0.0  310.0  3.611111   

          pres  ...        fwi  daily_severit

In [8]:
yvelines_data['date'] = pd.to_datetime(yvelines_data['date'], errors='coerce')
meteo['creneau'] = pd.to_datetime(meteo['creneau'], errors='coerce')

yvelines_data['centroid_lat'] = yvelines_data['centroid_lat'].astype(float)
yvelines_data['centroid_lon'] = yvelines_data['centroid_lon'].astype(float)
meteo['latitude'] = meteo['latitude'].astype(float)
meteo['longitude'] = meteo['longitude'].astype(float)

missing_dates = []
meteo_yvelines_pairs = []

for idx, row in yvelines_data.iterrows():
    date_yvelines = row['date']
    centroid_coords = (row['centroid_lat'], row['centroid_lon'])

    meteo_candidates = meteo[meteo['creneau'] == date_yvelines]
    print(f"Date : {date_yvelines}, Candidats : {len(meteo_candidates)}")

    if meteo_candidates.empty:
        missing_dates.append(row['hex_id'])
        continue

    meteo_candidates['distance'] = meteo_candidates.apply(
        lambda x: geodesic(centroid_coords, (x['latitude'], x['longitude'])).meters, axis=1
    )

    closest_match = meteo_candidates.loc[meteo_candidates['distance'].idxmin()]
    meteo_yvelines_pairs.append({'hex_id': row['hex_id'], 'id_meteo': closest_match['id_meteo']})

meteo_yvelines = pd.DataFrame(meteo_yvelines_pairs)
print(f"Dates manquantes : {len(missing_dates)}")

Date : 2017-01-16 00:00:00, Candidats : 67
Date : 2017-01-19 00:00:00, Candidats : 67
Date : 2017-01-20 00:00:00, Candidats : 67
Date : 2017-01-26 00:00:00, Candidats : 67
Date : 2017-01-27 00:00:00, Candidats : 67


<ipython-input-8-bb8305368680>:23: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  meteo_candidates['distance'] = meteo_candidates.apply(


Date : 2017-01-28 00:00:00, Candidats : 67
Date : 2017-02-01 00:00:00, Candidats : 67
Date : 2017-02-06 00:00:00, Candidats : 67
Date : 2017-02-06 00:00:00, Candidats : 67
Date : 2017-02-12 00:00:00, Candidats : 67
Date : 2017-02-15 00:00:00, Candidats : 67
Date : 2017-02-15 00:00:00, Candidats : 67
Date : 2017-02-23 00:00:00, Candidats : 67
Date : 2017-02-26 00:00:00, Candidats : 67
Date : 2017-02-26 00:00:00, Candidats : 67
Date : 2017-02-28 00:00:00, Candidats : 67
Date : 2017-03-12 00:00:00, Candidats : 67
Date : 2017-03-14 00:00:00, Candidats : 67
Date : 2017-03-17 00:00:00, Candidats : 67
Date : 2017-03-18 00:00:00, Candidats : 67
Date : 2017-03-21 00:00:00, Candidats : 67
Date : 2017-03-27 00:00:00, Candidats : 67
Date : 2017-03-27 00:00:00, Candidats : 67
Date : 2017-03-28 00:00:00, Candidats : 67
Date : 2017-03-29 00:00:00, Candidats : 67
Date : 2017-03-30 00:00:00, Candidats : 67
Date : 2017-04-02 00:00:00, Candidats : 67
Date : 2017-04-04 00:00:00, Candidats : 67
Date : 2017

In [9]:
for idx, row in meteo_yvelines.iterrows():
    hex_id = row['hex_id']

    yvelines_row = yvelines_data[yvelines_data['hex_id'] == hex_id].iloc[0]

    meteo_yvelines.at[idx, 'duree_minutes'] = yvelines_row['duree_minutes']
    meteo_yvelines.at[idx, 'date'] = yvelines_row['date']
    meteo_yvelines.at[idx, 'departement'] = yvelines_row['departement']
    meteo_yvelines.at[idx, 'geometry'] = yvelines_row['geometry']
    meteo_yvelines.at[idx, 'centroid_lon'] = yvelines_row['centroid_lon']
    meteo_yvelines.at[idx, 'centroid_lat'] = yvelines_row['centroid_lat']
    meteo_yvelines.at[idx, 'hour_sin'] = yvelines_row['hour_sin']
    meteo_yvelines.at[idx, 'hour_cos'] = yvelines_row['hour_cos']
    meteo_yvelines.at[idx, 'month_sin'] = yvelines_row['month_sin']
    meteo_yvelines.at[idx, 'month_cos'] = yvelines_row['month_cos']
    meteo_yvelines.at[idx, 'weekday_sin'] = yvelines_row['weekday_sin']
    meteo_yvelines.at[idx, 'weekday_cos'] = yvelines_row['weekday_cos']

    if (idx + 1) % 100 == 0:
        print(f"Ligne {idx + 1}/{len(meteo_yvelines)} mise à jour...")

print("Mise à jour terminée.")

Ligne 100/2866 mise à jour...
Ligne 200/2866 mise à jour...
Ligne 300/2866 mise à jour...
Ligne 400/2866 mise à jour...
Ligne 500/2866 mise à jour...
Ligne 600/2866 mise à jour...
Ligne 700/2866 mise à jour...
Ligne 800/2866 mise à jour...
Ligne 900/2866 mise à jour...
Ligne 1000/2866 mise à jour...
Ligne 1100/2866 mise à jour...
Ligne 1200/2866 mise à jour...
Ligne 1300/2866 mise à jour...
Ligne 1400/2866 mise à jour...
Ligne 1500/2866 mise à jour...
Ligne 1600/2866 mise à jour...
Ligne 1700/2866 mise à jour...
Ligne 1800/2866 mise à jour...
Ligne 1900/2866 mise à jour...
Ligne 2000/2866 mise à jour...
Ligne 2100/2866 mise à jour...
Ligne 2200/2866 mise à jour...
Ligne 2300/2866 mise à jour...
Ligne 2400/2866 mise à jour...
Ligne 2500/2866 mise à jour...
Ligne 2600/2866 mise à jour...
Ligne 2700/2866 mise à jour...
Ligne 2800/2866 mise à jour...
Mise à jour terminée.


In [10]:
for idx, row in meteo_yvelines.iterrows():
    id_meteo = row['id_meteo']

    meteo_row = meteo[meteo['id_meteo'] == id_meteo].iloc[0]

    for col in meteo.columns:
        if col not in ['longitude', 'latitude', 'id_meteo']:
            meteo_yvelines.at[idx, col] = meteo_row[col]

    if (idx + 1) % 100 == 0: 
        print(f"Ligne {idx + 1}/{len(meteo_yvelines)} mise à jour...")

print("Mise à jour terminée.")

Ligne 100/2866 mise à jour...
Ligne 200/2866 mise à jour...
Ligne 300/2866 mise à jour...
Ligne 400/2866 mise à jour...
Ligne 500/2866 mise à jour...
Ligne 600/2866 mise à jour...
Ligne 700/2866 mise à jour...
Ligne 800/2866 mise à jour...
Ligne 900/2866 mise à jour...
Ligne 1000/2866 mise à jour...
Ligne 1100/2866 mise à jour...
Ligne 1200/2866 mise à jour...
Ligne 1300/2866 mise à jour...
Ligne 1400/2866 mise à jour...
Ligne 1500/2866 mise à jour...
Ligne 1600/2866 mise à jour...
Ligne 1700/2866 mise à jour...
Ligne 1800/2866 mise à jour...
Ligne 1900/2866 mise à jour...
Ligne 2000/2866 mise à jour...
Ligne 2100/2866 mise à jour...
Ligne 2200/2866 mise à jour...
Ligne 2300/2866 mise à jour...
Ligne 2400/2866 mise à jour...
Ligne 2500/2866 mise à jour...
Ligne 2600/2866 mise à jour...
Ligne 2700/2866 mise à jour...
Ligne 2800/2866 mise à jour...
Mise à jour terminée.


In [14]:
firepoint_3 = firepoint_3.drop(columns=['geometry'], errors='ignore')

meteo_yvelines = meteo_yvelines.merge(firepoint_3, on='hex_id', how='left')

print(meteo_yvelines.head())

            hex_id  id_meteo  duree_minutes       date  \
0  871fb4440ffffff     25591           26.0 2017-01-16   
1  871fb6b73ffffff     25762           30.0 2017-01-19   
2  871fb478affffff     25851           24.0 2017-01-20   
3  871fb6b29ffffff     26231           76.0 2017-01-26   
4  871fb478affffff     26270           24.0 2017-01-20   

               departement                                           geometry  \
0  departement-78-yvelines  POLYGON ((1.838013394602329 48.94398261373458,...   
1  departement-78-yvelines  POLYGON ((1.828750965811647 48.680985964224924...   
2  departement-78-yvelines  POLYGON ((2.037816775415465 48.784120385066025...   
3  departement-78-yvelines  POLYGON ((1.760742884844425 48.71539708210374,...   
4  departement-78-yvelines  POLYGON ((2.037816775415465 48.784120385066025...   

   centroid_lon  centroid_lat  hour_sin  hour_cos  ...    water       tree  \
0      1.840467     48.931599 -0.965926 -0.258819  ...  0.00000  35.333079   
1      1

In [15]:
meteo_yvelines.head()

,hex_id,id_meteo,duree_minutes,date,departement,geometry,centroid_lon,centroid_lat,hour_sin,hour_cos,...,water,tree,grass,crops,shrub,flooded,built,bare,snow_y,sinister
0,871fb4440ffffff,25591,26.0,2017-01-16,departement-78-yvelines,"POLYGON ((1.838013394602329 48.94398261373458,...",1.840467,48.931599,-0.965926,-0.258819,...,0.00000,35.333079,4.203286,1.362884,10.164310,0.471278,26.557126,0.0,0.0,5.0
1,871fb6b73ffffff,25762,30.0,2017-01-19,departement-78-yvelines,POLYGON ((1.828750965811647 48.680985964224924...,1.831196,48.668582,-0.866025,0.500000,...,0.24244,71.672834,2.462677,0.038280,5.052954,0.000000,1.224959,0.0,0.0,2.0
2,871fb478affffff,25851,24.0,2017-01-20,departement-78-yvelines,POLYGON ((2.037816775415465 48.784120385066025...,2.040218,48.771733,-0.965926,-0.258819,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,871fb6b29ffffff,26231,76.0,2017-01-26,departement-78-yvelines,"POLYGON ((1.760742884844425 48.71539708210374,...",1.763205,48.702993,-0.965926,-0.258819,...,0.00000,68.445180,10.963963,0.063670,3.629186,0.000000,0.203744,0.0,0.0,0.0
4,871fb478affffff,26270,24.0,2017-01-20,departement-78-yvelines,POLYGON ((2.037816775415465 48.784120385066025...,2.040218,48.771733,-0.965926,-0.258819,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [20]:
meteo_yvelines_gdf = gpd.GeoDataFrame(meteo_yvelines, geometry='geometry')
meteo_yvelines_gdf.to_file('../datasets_par_departement/meteo_yvelines.geojson', driver='GeoJSON')